# **Eksploracyjna Analiza Danych (EDA): Zbiór Danych Po Tokenizacji**

---

### **Cel Analizy**

Niniejszy notatnik przedstawia kompleksową eksploracyjną analizę danych (EDA) dla przetworzonego i stokenizowanego zbioru danych, stworzonego na potrzeby projektu. Głównym celem jest dogłębne zrozumienie charakterystyki danych przed przystąpieniem do etapu treningu modelu.

Analiza skupia się na czterech kluczowych obszarach:
1.  **Struktura Zbioru Danych:** Zbadanie transformacji pełnotekstowych raportów korporacyjnych na fragmenty (ang. *chunks*) o stałej długości tokenów.
2.  **Ocena Jakości Podziału Danych:** Statystyczna weryfikacja integralności i zbalansowania zbiorów treningowego, walidacyjnego i testowego.
3.  **Dystrybucja Kryteriów ESG:** Analiza częstości występowania oraz współwystępowania sześciu kryteriów ML w celu identyfikacji potencjalnych wyzwań modelowania, takich jak niezbalansowanie klas.
4.  **Efektywność Tokenizacji:** Ocena, jak skutecznie wykorzystywane jest okno kontekstowe modelu Longformer (4096 tokenów).

Wnioski z tej analizy stanowią fundamentalne uzasadnienie dla podjętych decyzji metodologicznych oraz dla rzetelnej interpretacji ostatecznej wydajności modelu.

### **1.1. Konfiguracja i Wczytanie Danych**

Na początku importujemy niezbędne biblioteki i wczytujemy główną konfigurację projektu. Zapewnia to spójność analizy z parametrami całego potoku przetwarzania.

In [ ]:
# --- Główne Biblioteki ---
import json
import numpy as np
import pandas as pd
from collections import Counter
from pathlib import Path

# --- Obsługa Danych (Hugging Face) ---
from datasets import load_from_disk, concatenate_datasets

# --- Wizualizacja ---
import plotly.express as px
import plotly.graph_objects as go
import plotly.figure_factory as ff
import plotly.io as pio

# --- Analiza Statystyczna ---
from scipy.stats import chi2_contingency, ks_2samp
from statsmodels.stats.multitest import multipletests

# --- Ustawienia Notatnika ---
import warnings
warnings.filterwarnings('ignore')
pio.templates.default = "plotly_white"

# Kryteria ML do użytku w wykresach i analizie.
CRITERIA_NAMES = [
    "1",
    "2",
    "4",
    "6",
    "7",
    "8",
]

np.random.seed(42)
RANDOM_SEED = 42

### Kryteria

#### 1. Plan Transformacji Klimatycznej (ESRS E1-1)
*   **Opis:** Kryterium ocenia, czy raport przedstawia plan transformacji klimatycznej zintegrowany ze strategią biznesową. Model koncentruje się na fundamentalnym wymogu, jakim jest wykazanie istnienia i integracji planu, co jest podstawą dla dalszych, bardziej szczegółowych analiz zgodności z ESRS.

#### 2. Identyfikacja i Zarządzanie Ryzykami (ESRS E1-2)
*   **Opis:** Kryterium ocenia, czy oprócz samej identyfikacji ryzyk i szans klimatycznych, raport opisuje również **aktywne procesy zarządcze** służące ich mitygacji. Celem jest odróżnienie pasywnej świadomości od wdrożonych, działających polityk.

#### 4. Definicja Granic Konsolidacji (ESRS E1-6 Metodyka)
*   **Opis:** Kryterium ocenia, czy raport w sposób transparentny definiuje granice organizacyjne dla raportowanych emisji, wraz z uzasadnieniem ewentualnych wyłączeń. Jest to fundamentalny aspekt metodologiczny, kluczowy dla porównywalności i wiarygodności danych.

#### 6. Raportowanie Danych Historycznych (ESRS 1 + ESRS E1)
*   **Opis:** Kryterium ocenia, czy raport dostarcza dane historyczne za minimum 3 lata, co umożliwia rzetelną analizę trendów. Wymóg prezentacji danych porównawczych jest zasadą jakościową, a próg 3 lat pozwala na odróżnienie minimalnego raportowania od wiarygodnej prezentacji dynamiki zmian.

#### 7. Ujawnienie Wskaźników Intensywności (ESRS E1-6)
*   **Opis:** Kryterium weryfikuje, czy raport, oprócz podania wartości bezwzględnych emisji, przedstawia również wskaźniki intensywności (np. emisje na jednostkę przychodu lub produkcji). Pozwala to na ocenę eko-efektywności i porównywalność między firmami o różnej skali działalności.

#### 8. Wiarygodność Zdefiniowanych Celów (ESRS E1-4)
*   **Opis:** Kryterium ocenia, czy strategia redukcyjna jest kompletna, tzn. czy łączy zdefiniowane cele redukcyjne (np. redukcja emisji o X% do roku Y) z konkretnym planem działań, który ma do nich doprowadzić. Jest to najbardziej rygorystyczne kryterium, mające na celu odróżnienie wiarygodnych strategii od potencjalnego greenwashingu.

In [ ]:
# Wczytaj konfigurację projektu z głównego katalogu
with open("../config.json") as f:
    cfg = json.load(f)
print("✅ Konfiguracja wczytana pomyślnie.")

dataset_path = Path("../" + cfg.get("tokenizer_output_path", "data/processed/tokenized_dataset"))
dataset = load_from_disk(str(dataset_path))
print(f"✅ Stokenizowany zbiór danych wczytany z: '{dataset_path}'")

✅ Konfiguracja wczytana pomyślnie.
✅ Stokenizowany zbiór danych wczytany z: '..\data\processed\tokenized_dataset'


<br>

---

### **1.2. Przegląd Zbioru Danych: Od Dokumentów do Fragmentów**

Kluczowym krokiem w przygotowaniu danych było zastosowanie techniki "przesuwnego okna" (*sliding window*) w celu posegmentowania długich raportów korporacyjnych na fragmenty o długości 4096 tokenów. Proces ten znacząco zwiększył liczbę próbek treningowych. Poniższa tabela podsumowuje wynik tej transformacji i potwierdza integralność stratyfikowanego podziału na zbiory w proporcji 80/10/10.

In [ ]:
# Połącz wszystkie podzbiory w celu uzyskania ogólnego przeglądu
full_dataset = concatenate_datasets([dataset['train'], dataset['validation'], dataset['test']])

# --- Obliczanie Kluczowych Statystyk ---
total_chunks = len(full_dataset)
unique_doc_ids = full_dataset.unique("doc_id")
total_docs = len(unique_doc_ids)

# Oblicz statystyki dla każdego podzbioru
split_stats = {}
for split_name in dataset.keys():
    split_data = dataset[split_name]
    split_docs = len(split_data.unique("doc_id"))
    split_chunks = len(split_data)
    split_stats[split_name] = {
        "docs": split_docs,
        "chunks": split_chunks,
        "docs_pct": (split_docs / total_docs) * 100 if total_docs > 0 else 0,
        "chunks_pct": (split_chunks / total_chunks) * 100 if total_chunks > 0 else 0,
        "avg_chunks_per_doc": split_chunks / split_docs if split_docs > 0 else 0
    }

header = ["<b>Metryka</b>", "<b>Zbiór Treningowy</b>", "<b>Zbiór Walidacyjny</b>", "<b>Zbiór Testowy</b>", "<b>Suma</b>"]
rows = [
    ["Dokumenty", 
      f"{split_stats['train']['docs']} ({split_stats['train']['docs_pct']:.1f}%)", 
      f"{split_stats['validation']['docs']} ({split_stats['validation']['docs_pct']:.1f}%)", 
      f"{split_stats['test']['docs']} ({split_stats['test']['docs_pct']:.1f}%)", 
      f"<b>{total_docs}</b>"],
    ["Fragmenty (Próbki Treningowe)", 
      f"{split_stats['train']['chunks']:,}", 
      f"{split_stats['validation']['chunks']:,}", 
      f"{split_stats['test']['chunks']:,}", 
      f"<b>{total_chunks:,}</b>"],
    ["Śr. liczba fragmentów na dokument", 
      f"{split_stats['train']['avg_chunks_per_doc']:.2f}", 
      f"{split_stats['validation']['avg_chunks_per_doc']:.2f}", 
      f"{split_stats['test']['avg_chunks_per_doc']:.2f}", 
      f"<b>{(total_chunks / total_docs):.2f}</b>"]
]

fig = go.Figure(data=[go.Table(
    header=dict(values=header, fill_color='#264653', line_color='darkslategray',
                align='center', font=dict(color='white', size=14)),
    cells=dict(values=list(zip(*rows)), fill_color='#F4A261', line_color='darkslategray',
                align='center', font=dict(color='black', size=12), height=30)
)])
fig.update_layout(
    title_text="<b>Przegląd Zbioru Danych: Dokumenty vs. Fragmenty</b>", 
    title_x=0.5,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig.show()

#### **Kluczowe Wnioski:**
*   Oryginalne **393** raporty korporacyjne zostały przetworzone na **7061** próbek treningowych (fragmentów).
*   Średnio, każdy dokument został podzielony na około **18 fragmentów**, co dostarcza modelowi bogatych, skonteneryzowanych danych.
*   Podział na poziomie dokumentów został zachowany w proporcjach około **80% (trening), 10% (walidacja) i 10% (test)**, co potwierdza poprawność procesu stratyfikacji.

<br>

---

### **2. Ocena Jakości Podziału Danych**

Wiarygodna ocena modelu zależy od porównywalności zbiorów treningowego, walidacyjnego i testowego. Niewłaściwie skonstruowany podział mógłby prowadzić do mylących metryk wydajności. W tej sekcji weryfikujemy statystycznie, czy nasz stratyfikowany proces podziału stworzył zrównoważone i reprezentatywne podzbiory.

Testujemy dwie kluczowe hipotezy zerowe:
- **H₀₁:** Dystrybucje długości dokumentów (mierzone liczbą fragmentów) są identyczne we wszystkich trzech zbiorach.
- **H₀₂:** Dystrybucje etykiet pozytywnych dla każdego z 6 kryteriów ESG są identyczne we wszystkich trzech zbiorach.

#### **2.1. Dystrybucja Długości Dokumentów**

Wizualnie sprawdzamy i porównujemy rozkład długości dokumentów (tj. na ile fragmentów każdy dokument został podzielony) w poszczególnych zbiorach.

In [ ]:
# --- Przygotuj dane do wykresu ---
doc_chunk_counts = Counter(full_dataset['doc_id'])

length_df = pd.DataFrame({
    "doc_id": list(doc_chunk_counts.keys()),
    "chunk_count": list(doc_chunk_counts.values())
})

# Przypisz każdy dokument do jego oryginalnego podziału
train_docs = set(dataset['train'].unique("doc_id"))
val_docs = set(dataset['validation'].unique("doc_id"))
test_docs = set(dataset['test'].unique("doc_id"))

def get_split(doc_id):
    if doc_id in train_docs:
        return "Treningowy"
    if doc_id in val_docs:
        return "Walidacyjny"
    return "Testowy"

length_df['split'] = length_df['doc_id'].apply(get_split)

fig = px.violin(
    length_df,
    x="split",
    y="chunk_count",
    color="split",
    box=True,
    points=False,
    labels={
        "split": "Podział Zbioru Danych",
        "chunk_count": "Liczba Fragmentów na Dokument"
    },
    title="<b>Dystrybucja Długości Dokumentów w Podziałach</b>",
    color_discrete_map={
        "Treningowy": "#1f77b4",
        "Walidacyjny": "#ff7f0e",
        "Testowy": "#2ca02c"
    }
)

fig.update_layout(
    title_x=0.5,
    showlegend=False
)

fig.show()

Wizualnie dystrybucje wyglądają bardzo podobnie. Mediany (linia wewnątrz pudełek) są niemal identyczne, a zakresy międzykwartylowe (pudełka) w znacznym stopniu się pokrywają. Jest to mocny sygnał, że podział jest dobrze zrównoważony pod względem długości dokumentów.

#### **2.2. Dystrybucja Etykiet**

Następnie weryfikujemy, czy częstość występowania każdego kryterium ESG jest spójna między podziałami. Jest to test o kluczowym znaczeniu, ponieważ zapewnia, że model jest trenowany i ewaluowany na podobnych wzorcach danych.

In [ ]:
# --- Agreguj etykiety do poziomu dokumentu ---
# Dokument ma etykietę pozytywną, jeśli którykolwiek z jego fragmentów ma etykietę pozytywną.
# Jest to równoznaczne z wzięciem max(), ponieważ etykiety to 0 lub 1.
doc_level_labels_df = full_dataset.to_pandas()

# Przetwórz kolumnę etykiet: przekonwertuj na tablice numpy i przeprowadź agregację
def aggregate_labels(group):
    # Złóż wszystkie tablice etykiet i weź maksimum dla każdego elementu
    label_arrays = np.stack(group['labels'].values)
    return np.maximum.reduce(label_arrays)

# Grupuj po doc_id i agreguj etykiety
doc_level_labels = doc_level_labels_df.groupby('doc_id').apply(aggregate_labels).reset_index()
doc_level_labels.columns = ['doc_id', 'labels']

# Przekonwertuj etykiety z powrotem na DataFrame z odpowiednimi nazwami kolumn
labels_expanded = pd.DataFrame(doc_level_labels['labels'].tolist(), columns=CRITERIA_NAMES)
doc_level_labels_df = pd.concat([doc_level_labels[['doc_id']], labels_expanded], axis=1)
doc_level_labels_df.set_index('doc_id', inplace=True)

doc_level_labels_df['split'] = doc_level_labels_df.index.to_series().apply(get_split)

# --- Oblicz proporcję etykiet pozytywnych dla każdego kryterium w każdym podziale ---
proportions = doc_level_labels_df.groupby('split')[CRITERIA_NAMES].mean().reset_index()
proportions_melted = proportions.melt(
    id_vars='split', 
    var_name='criterion', 
    value_name='proportion'
)

fig = px.bar(
    proportions_melted,
    x="criterion",
    y="proportion",
    color="split",
    barmode="group",
    labels={
        "criterion": "Kryterium ESG",
        "proportion": "Proporcja Etykiet Pozytywnych",
        "split": "Podział Zbioru Danych"
    },
    title="<b>Dystrybucja Kryteriów ESG w Podziałach</b>",
    text_auto=".1%",
    color_discrete_map={
        'Treningowy': '#1f77b4',
        'Walidacyjny': '#ff7f0e',
        'Testowy': '#2ca02c'
    }
)

fig.update_layout(
    title_x=0.5,
    yaxis_tickformat=".0%",
    font=dict(size=12),
    legend=dict(
        orientation="h",
        yanchor="bottom",
        y=1.02,
        xanchor="right",
        x=1
    )
)

fig.update_traces(
    textangle=0, 
    textposition="outside", 
    cliponaxis=False
)

fig.show()

Wykres pokazuje, że zastosowana stratyfikacja generalnie zadziałała poprawnie, utrzymując zbliżone proporcje etykiet dla większości kryteriów.
Widoczne są jednak rozbieżności, zwłaszcza dla kryterium 6. 
Są one oczekiwanym efektem ubocznym procesu przetwarzania danych: stratyfikacja była przeprowadzana na poziomie całych dokumentów, ale późniejszy podział na fragmenty (chunking) w połączeniu z różną długością raportów wpłynął na finalną proporcję na poziomie fragmentów.

#### **2.3. Weryfikacja Statystyczna**

Aby formalnie potwierdzić nasze obserwacje wizualne, przeprowadzamy testy statystyczne. Ponieważ wykonujemy wiele testów jednocześnie, musimy zastosować korektę, aby uniknąć inflacji błędu pierwszego rodzaju (wyników fałszywie pozytywnych). Użyjemy **korekty Bonferroniego**, konserwatywnej metody dostosowującej wartości p.

-   **Test Kołmogorowa-Smirnowa (KS):** Porównuje dystrybucje długości dokumentów.
-   **Test Chi-kwadrat (χ²):** Porównuje dystrybucje etykiet kategorialnych.

Dla wszystkich testów przyjmujemy poziom istotności (α) **0.05**. Skorygowana wartość p > 0.05 oznacza, że nie ma podstaw do odrzucenia hipotezy zerowej, a więc dystrybucje są statystycznie podobne.

In [ ]:
train_lengths = length_df[length_df['split'] == 'Treningowy']['chunk_count']
val_lengths = length_df[length_df['split'] == 'Walidacyjny']['chunk_count']
test_lengths = length_df[length_df['split'] == 'Testowy']['chunk_count']

train_labels = doc_level_labels_df[doc_level_labels_df['split'] == 'Treningowy'][CRITERIA_NAMES]
val_labels = doc_level_labels_df[doc_level_labels_df['split'] == 'Walidacyjny'][CRITERIA_NAMES]
test_labels = doc_level_labels_df[doc_level_labels_df['split'] == 'Testowy'][CRITERIA_NAMES]

results = []
test_names = []
p_values = []

# Test KS dla długości dokumentów (porównanie Treningowy vs. Testowy jest najważniejsze)
test_names.append("Długości Dokumentów (Test KS)")
ks_stat, ks_p = ks_2samp(train_lengths, test_lengths)
p_values.append(ks_p)

# Test Chi-kwadrat dla każdej dystrybucji etykiet
for criterion in CRITERIA_NAMES:
    test_names.append(f"Kryterium {criterion} (Test χ²)")
    contingency_table = pd.crosstab(doc_level_labels_df['split'], doc_level_labels_df[criterion])
    chi2, p_val, _, _ = chi2_contingency(contingency_table)
    p_values.append(p_val)
    
# Zastosuj korektę Bonferroniego dla wielokrotnych porównań
reject, corrected_p_values, _, _ = multipletests(p_values, alpha=0.05, method='bonferroni')

results_data = {
    "Testowany Obiekt": test_names,
    "Surowa wartość p": [f"{p:.3f}" for p in p_values],
    "Skorygowana wartość p": [f"{cp:.3f}" for cp in corrected_p_values],
    "Wniosek": ["✅ Zaliczone" if not r else "❌ Niezaliczone" for r in reject]
}
results_df = pd.DataFrame(results_data)

header = [f"<b>{col}</b>" for col in results_df.columns]
fig = go.Figure(data=[go.Table(
    header=dict(values=header, fill_color='#264653', line_color='darkslategray',
                align=['left', 'center', 'center', 'center'], font=dict(color='white', size=14)),
    cells=dict(values=results_df.T.values, align=['left', 'center', 'center', 'center'], 
                font=dict(size=12), height=30)
)])
fig.update_layout(
    title_text="<b>Wyniki Testów Statystycznych dla Jednorodności Podziałów (z Korektą Bonferroniego)</b>", 
    title_x=0.5,
    margin=dict(l=20, r=20, t=60, b=20)
)
fig.show()

#### **Wniosek z Oceny Podziału:**
Nawet po zastosowaniu rygorystycznej korekty Bonferroniego, **wszystkie skorygowane wartości p pozostają znacznie powyżej poziomu istotności 0.05.** Daje to silne podstawy statystyczne do stwierdzenia, że:
-   **Nie ma statystycznie istotnych różnic** w dystrybucjach długości dokumentów ani częstości etykiet pomiędzy zbiorami treningowym, walidacyjnym i testowym.
-   Zastosowany stratyfikowany podział był **wysoce skuteczny i solidny**, tworząc jednorodne podzbiory.
-   Metryki wydajności uzyskane podczas ewaluacji modelu na tym zbiorze testowym będą **wiarygodne i reprezentatywne**.

<br>

---

### **3. Analiza Dystrybucji Kryteriów ESG**

Po potwierdzeniu jakości podziału danych, przechodzimy do charakterystyki samych kryteriów ESG. Zrozumienie ich częstości, niezbalansowania i wzajemnych relacji jest kluczowe dla zaprojektowania skutecznej strategii treningowej.

#### **3.1. Częstość i Niezbalansowanie Etykiet**

Analizujemy częstość występowania każdego z 6 kryteriów ML w całym zbiorze 393 dokumentów. Pozwoli to zidentyfikować, które kryteria są powszechne, a które rzadkie, co unaoczni główne wyzwanie badawcze: niezbalansowanie klas.

In [ ]:
# --- Oblicz częstość i proporcje etykiet na poziomie dokumentu ---
doc_labels_df = doc_level_labels_df.drop(columns=['split']) # Użyj df z poprzedniego kroku
label_frequencies = doc_labels_df.sum()
label_proportions = doc_labels_df.mean()
total_docs = len(doc_labels_df)

def get_bar_color(proportion):
    if proportion > 0.4: return '#2a9d8f'  # Dobrze zbalansowane
    if proportion > 0.2: return '#e9c46a'  # Umiarkowane niezbalansowanie
    return '#e76f51'  # Poważne niezbalansowanie

colors = [get_bar_color(p) for p in label_proportions]

fig = go.Figure(go.Bar(
    x=label_proportions.index,
    y=label_proportions.values,
    text=[f'{freq}<br>({prop:.1%})' for freq, prop in zip(label_frequencies, label_proportions)],
    textposition='outside',
    marker_color=colors,
    hovertemplate='<b>%{x}</b><br>Proporcja: %{y:.1%}<br>Dokumenty: %{text.split("<br>")[0]}<extra></extra>'
))

# Dodaj linie referencyjne dla kontekstu
fig.add_hline(y=0.5, line_dash="dash", line_color="gray", annotation_text="Idealna Równowaga (50%)")
fig.add_hline(y=0.2, line_dash="dot", line_color="#e76f51", annotation_text="Próg Niezbalansowania (20%)")

fig.update_layout(
    title_text="<b>Częstość i Niezbalansowanie Kryteriów ESG</b>",
    title_x=0.5,
    xaxis_title="Kryterium ESG",
    yaxis_title="Proporcja Etykiet Pozytywnych",
    yaxis_tickformat=".0%",
    height=500,
    xaxis_tickangle=-45
)
fig.show()

#### **Kluczowe Wnioski dot. Niezbalansowania Klas:**

*   **Zróżnicowana Częstość:** Kryteria wykazują szeroki zakres częstości, od **Kryterium 1 (Plan Transformacji)** obecnego w 59.0% dokumentów, do **Kryterium 8 (Wiarygodność Celów)** występującego w zaledwie 16.3%.
*   **Silne Niezbalansowanie:** **Kryterium 8** jest silnie niezbalansowane, ze stosunkiem klas pozytywnych do negatywnych wynoszącym ok. 1:5. Jest to nasze najtrudniejsze kryterium, wymagające specjalnego podejścia podczas treningu.
*   **Umiarkowane Niezbalansowanie:** Kryteria **2, 6 i 7** również wykazują umiarkowane niezbalansowanie, spadając poniżej progu 40%.
*   **Uzasadnienie dla `class_weight`:** Obserwowane niezbalansowanie silnie uzasadnia użycie ważonej funkcji straty (takiej jak `class_weight='balanced'`) w procedurze treningowej, aby zapobiec ignorowaniu przez model rzadkich klas pozytywnych. Potwierdza również wybór **metryki F1-score** jako głównego wskaźnika oceny, ponieważ jest ona bardziej informatywna niż trafność (accuracy) na zbiorach niezbalansowanych.

#### **3.2. Współwystępowanie Kryteriów ESG**

Następnie badamy relacje między kryteriami. Czy firmy, które spełniają jedno kryterium, mają tendencję do spełniania innych? Mapa korelacji pomaga odpowiedzieć na to pytanie.

In [ ]:
# --- Oblicz macierz korelacji ---
corr_matrix = doc_level_labels_df[CRITERIA_NAMES].corr(method='pearson')

mask = np.triu(np.ones_like(corr_matrix, dtype=bool))

# --- Utwórz mapę ciepła ---
fig = go.Figure(data=go.Heatmap(
    z=corr_matrix.where(~mask),
    x=corr_matrix.columns,
    y=corr_matrix.columns,
    colorscale='RdBu',
    zmid=0,
    zmin=-1, zmax=1,
    hovertemplate='<b>Korelacja</b><br>%{y} - %{x}: %{z:.2f}<extra></extra>'
))

for i in range(len(corr_matrix.columns)):
    for j in range(i):
        fig.add_annotation(
            text=f"{corr_matrix.iloc[i, j]:.2f}",
            x=j,
            y=i,
            xref="x", yref="y",
            showarrow=False,
            font=dict(color="white" if abs(corr_matrix.iloc[i, j]) > 0.4 else "black")
        )
        
fig.update_layout(
    title_text='<b>Macierz Współwystępowania Kryteriów ESG (Współczynnik Fi)</b>',
    title_x=0.5,
    height=600,
    width=650,
    xaxis_showgrid=False,
    yaxis_showgrid=False,
    xaxis=dict(tickmode='linear'),
    yaxis_autorange='reversed'
)

fig.show()

#### **Kluczowe Wnioski dot. Współwystępowania:**

*   **Wyłącznie Pozytywne Korelacje:** Zgodnie z oczekiwaniami, wszystkie korelacje są dodatnie. Firma, która aktywnie raportuje na jeden zaawansowany temat ESG, jest bardziej skłonna do raportowania na inne.
*   **Najsilniejsza Korelacja:** Najsilniejszy związek (**0.52**) występuje między **Kryterium 4 (Granice Konsolidacji)** a **Kryterium 7 (Wskaźniki Intensywności)**. Ma to uzasadnienie merytoryczne: zdefiniowanie jasnych granic jest warunkiem wstępnym do obliczenia miarodajnych wskaźników intensywności.
*   **Umiarkowane Klastry:** Widoczne są klastry umiarkowanie skorelowanych kryteriów (np. 4, 6 i 7), co sugeruje, że te metodologiczne aspekty raportowania emisji są często adresowane łącznie.
*   **Brak Redundancji:** Co istotne, nie występują skrajnie wysokie korelacje (np. > 0.8), co wskazuje, że **każde kryterium wychwytuje unikalny i odrębny aspekt raportowania ESG**. Potwierdza to, że wszystkie 6 kryteriów jest wartościowych i nieredundantnych dla naszego zadania klasyfikacyjnego.

<br>

---

### **4. Analiza Efektywności Tokenizacji**

Ostatnim krokiem naszej analizy jest weryfikacja technicznej efektywności strategii tokenizacji i podziału na fragmenty. Ponieważ używamy modelu o stałym rozmiarze okna kontekstowego (Longformer z 4096 tokenami), kluczowe jest upewnienie się, że efektywnie wykorzystujemy tę pojemność. Zmarnowana przestrzeń (spowodowana nadmiernym dopełnianiem, ang. *padding*) może prowadzić do nieefektywnego treningu i utraty informacji.


In [ ]:
# --- Analiza długości input_ids na próbce zbioru danych ---
# Używamy próbki dla wydajności.
sample_size = min(2000, len(full_dataset))
sample_indices = np.random.choice(len(full_dataset), sample_size, replace=False)
sampled_dataset = full_dataset.select(sample_indices)

# Rzeczywista długość sekwencji to suma jej maski uwagi (attention mask)
sequence_lengths = [sum(mask) for mask in sampled_dataset['attention_mask']]

avg_length = np.mean(sequence_lengths)
max_length = 4096
utilization_pct = (avg_length / max_length) * 100
full_chunks_pct = (sum(1 for length in sequence_lengths if length == max_length) / len(sequence_lengths)) * 100

print("="*50)
print("RAPORT EFEKTYWNOŚCI TOKENIZACJI")
print("="*50)
print(f"➢ Średnia długość sekwencji:       {avg_length:.1f} / {max_length} tokenów")
print(f"✅ Średnie wykorzystanie kontekstu:   {utilization_pct:.1f}%")
print(f"💯 Fragmenty o pełnej pojemności (4096): {full_chunks_pct:.1f}%")
print("="*50)

RAPORT EFEKTYWNOŚCI TOKENIZACJI
➢ Średnia długość sekwencji:       4012.4 / 4096 tokenów
✅ Średnie wykorzystanie kontekstu:   98.0%
💯 Fragmenty o pełnej pojemności (4096): 95.0%


#### **Wniosek:**

Analiza potwierdza sukces naszej strategii tokenizacji. Przy **średnim wykorzystaniu kontekstu na poziomie ok. 98%**, doskonale zagospodarowujemy pojemność modelu Longformer. Ta wysoka efektywność waliduje nasz dobór parametrów `stride` i `max_length` w potoku przygotowania danych, zapewniając, że model otrzymuje gęste, bogate w informacje dane wejściowe, co jest optymalne dla skutecznego uczenia.